In [ ]:
# Adnan Yousaf
# i191742@nu.edu.pk

In [153]:
import re

# Reading the text
with open('/content/urdu-corpus.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Original Text:")
print(text)

Original Text:
گزشتہ کئی سالوں سے مختلف بحران آتے جاتے رہے لیکن حالیہ آٹا ، چینی سمیت دیگر بحران اچانک پید ا ہوئے اور ان پر جے آئی ٹی تشکیل دے دیں گئیں تاکہ عوام کو ریلیف دیا جاسکے دوسری جانب بجلی ، گیس ، پانی سمیت دیگر بلوں میں کئی سو گنا اضافہ کردیا گیا ،صوبائی و وفاقی وزراء نے اپنے اپنے ایوانوں بحران کے ذمہ دار عناصر کو بے نقاب کرنے کے بجائے سب اچھاہے کی رپورٹ پیش کیں ساتھ ہی اورنگزیب کی طرح جواب دیا، حقائق کی ’’قبر ذرا گہری کھودنا!‘‘ تاریخی گواہ ہے بقول ماہرین ، مبصرین اور صحافیوں کا کہنا اور لکھنا ہے کہ پگڑی بدل بھائی کی رسم کی آڑ میں نادر شاہ نے محمد شاہ رنگیلا سے کوہِ نور ہیرا حاصل کیا تھا 12 مئی 1739 کی شام دہلی میں زبردست چہل پہل، شاہجہان آباد میں چراغاں اور لال قلعے میں جشن کا سماں ہے غریبوں میں شربت، پان اور کھانا تقسیم کیا جا رہا ہے، فقیروں، گداؤں کو جھولی بھر بھر کر روپے عطا ہو رہے ہیں آج دربار میں ایرانی بادشاہ نادر شاہ کے سامنے مغلیہ سلطنت کے 13ویں تاجدار محمد شاہ بیٹھے ہیں، لیکن اس وقت ان کے سر پر شاہی تاج نہیں ہے، کیوں نادر شاہ نے ڈھائی ماہ قبل ان سے سلطنت چھین لی تھی 

Excess spaces are eliminated from the text by normalizing it. This is an important
preprocessing step since irregular spacing frequently causes sentence boundaries to be
misinterpreted when working with huge datasets. Regular expressions are used by the
normalize_spaces function to replace numerous spaces with a single space.

In [168]:
# Normalizing spaces
def normalize_spaces(text):
    #extra spaces removal
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

Sentence segmentation is handled by the code by recognizing common punctuation and
Urdu sentence-ending terms, like ","تھی ","ہے and the Urdu full stop "."۔ Based on these signals,
the segmentation procedure makes sure that the sentences are separated appropriately.

In [169]:
# sentence segmentation
def urdu_sentence_segmentation(text):
    text = normalize_spaces(text)

    #common urdu sentence-ending words
    end_words = [
        'ہے', 'ہیں', 'تھا', 'تھی', 'تھے', 'تھیں', 'گی', 'گے', 'دی', 'دیے', 'دیا', 'کیا', 'کئے', 'کرے',
        'رہا', 'رہی', 'رہے', 'دی۔', 'ہے۔', 'ہیں۔', 'تھا۔', 'تھی۔', 'تھے۔', 'تھیں۔', 'گی۔', 'گے۔', 'دی۔',
        'دیا۔', 'کیا۔', 'کئے۔', 'کرے۔', 'رہا۔', 'رہی۔', 'رہے۔'
    ]

    #adding space after sentence-ending punctuation if its missing
    text = re.sub(r'([۔!?])([^ ])', r'\1 \2', text)
    # text to words
    words = text.split()

    sentences = []
    current_sentence = ''

    for i, word in enumerate(words):
        current_sentence += word + ' '
        # Remove punctuation for matching
        stripped_word = re.sub(r'[^\wٔءاأإآؤئںہھۀےۓۃھ]', '', word)

        if stripped_word in end_words or word.endswith('۔'):
            current_sentence = current_sentence.strip()
            sentences.append(current_sentence)
            current_sentence = ''

    #adding any remaining text as the last sentence
    if current_sentence.strip():
        sentences.append(current_sentence.strip())

    return sentences

# Segment the text
segmented_sentences = urdu_sentence_segmentation(text)

# print("segmented Sentences:")
# for idx, sentence in enumerate(segmented_sentences):
#     print(f"Sentence {idx+1}: {sentence}")

1. Encoder: Converts a list of sentences into tokens using the tokenizer.

2. Decoder: Converts the tokens back into sentences, reconstructing the original text.


In [170]:
#Encode and Decode Functions

#urdu punctuation marks and special characters
urdu_punctuation = '،؛۔!?\'"()[]{}<>:؛—…‘’“”'

#Tokenizer
def custom_tokenize(sentence):
    tokens = []
    current_token = ''
    for char in sentence:
        if char in urdu_punctuation or char.isspace():
            if current_token != '':
                tokens.append(current_token)
                current_token = ''
            if not char.isspace():
                tokens.append(char)
        else:
            current_token += char
    if current_token != '':
        tokens.append(current_token)
    return tokens

#Encoder
def custom_encode(sentences):
    encoded_sentences = []
    for sentence in sentences:
        tokens = custom_tokenize(sentence)
        encoded_sentences.append(tokens)
    return encoded_sentences

#Decoder
def custom_decode(encoded_sentences):
    decoded_sentences = []
    for tokens in encoded_sentences:
        sentence = ''
        skip_space = False
        for token in tokens:
            if token in urdu_punctuation:
                if token in '‘’“”""' and not sentence.endswith(' '):
                    #don't add space before quotation marks
                    sentence += token
                    skip_space = True
                else:
                    sentence += token
            else:
                if not skip_space and sentence and not sentence.endswith(' '):
                    sentence += ' '
                sentence += token
                skip_space = False
        decoded_sentences.append(sentence.strip())
    return decoded_sentences

In [171]:
#calling encoder to encode segmented sentences
encoded_sentences = custom_encode(segmented_sentences)
print("\nEncoded Sentences:")
for idx, tokens in enumerate(encoded_sentences):
    print(f"Sentence {idx+1} Tokens: {tokens}")

Streaming output truncated to the last 5000 lines.
Sentence 77729 Tokens: ['سپریم', 'کورٹ', 'کی', 'تین', 'رکنی', 'بینچ', 'نے', 'کہا', 'ہے']
Sentence 77730 Tokens: ['کہ', 'سیاسی', 'معاملات', 'پر', 'کورٹ', 'فیصلہ', 'نہیں', 'کر', 'سکتی', '۔']
Sentence 77731 Tokens: ['ساوتھ', 'ایشین', 'وائر', 'کے', 'مطابق', 'فیصلے', 'میں', 'کہا', 'گیا', 'ہے']
Sentence 77732 Tokens: ['کہ', 'جینے', 'کے', 'حق', 'کی', 'حفاظت', 'کی', 'جانی', 'چاہیے', '۔']
Sentence 77733 Tokens: ['سپریم', 'کورٹ', 'نے', 'اپنے', 'فیصلے', 'میں', 'واضح', 'انداز', 'میں', 'کہا', 'کہ', 'انٹرنیٹ', 'پر', 'پابندی', 'لگانا', 'مناسب', 'نہیں', 'ہے', '۔']
Sentence 77734 Tokens: ['یہ', 'جمہوری', 'ملک', 'ہے', '،']
Sentence 77735 Tokens: ['یہاں', 'ہم', 'کسی', 'کو', 'اس', 'طرح', 'نہیں', 'رکھ', 'سکتے', '۔']
Sentence 77736 Tokens: ['ساوتھ', 'ایشین', 'وائر', 'کے', 'مطابق', 'عدالت', 'عظمی', 'کا', 'کہنا', 'ہے']
Sentence 77737 Tokens: ['کہ', 'انٹرنیٹ', 'پر', 'پابندی', 'اظہار', 'رائے', 'پر', 'پابندی', 'لگانے', 'کے', 'مترادف', 'ہے', '۔']
Sentence 77738 T

In [172]:
#decoding encoded sentences
decoded_sentences = custom_decode(encoded_sentences)
print("\nDecoded Sentences:")
for idx, sentence in enumerate(decoded_sentences):
    print(f"Sentence {idx+1}: {sentence}")

Streaming output truncated to the last 5000 lines.
Sentence 77729: سپریم کورٹ کی تین رکنی بینچ نے کہا ہے
Sentence 77730: کہ سیاسی معاملات پر کورٹ فیصلہ نہیں کر سکتی۔
Sentence 77731: ساوتھ ایشین وائر کے مطابق فیصلے میں کہا گیا ہے
Sentence 77732: کہ جینے کے حق کی حفاظت کی جانی چاہیے۔
Sentence 77733: سپریم کورٹ نے اپنے فیصلے میں واضح انداز میں کہا کہ انٹرنیٹ پر پابندی لگانا مناسب نہیں ہے۔
Sentence 77734: یہ جمہوری ملک ہے،
Sentence 77735: یہاں ہم کسی کو اس طرح نہیں رکھ سکتے۔
Sentence 77736: ساوتھ ایشین وائر کے مطابق عدالت عظمی کا کہنا ہے
Sentence 77737: کہ انٹرنیٹ پر پابندی اظہار رائے پر پابندی لگانے کے مترادف ہے۔
Sentence 77738: لوگوں کے حقوق نہیں چھینے جانے چاہیے۔
Sentence 77739: سات دنوں کے اندر دفعہ 144 پر جائزہ لیا جانا چاہیے۔
Sentence 77740: حکومت دلائل کو سپریم کورٹ نے رد کر دیا۔
Sentence 77741: کہیں بھی دفعہ144 لگائی جائے تو اسے غیرمعینہ نہیں کیا
Sentence 77742: جاسکتا۔
Sentence 77743: غیر معمولی حالات میں ہی اس کا نفاذ کیا
Sentence 77744: جاسکتا ہے۔
Sentence 77745: اس دفعہ کا استع

In [173]:
# Verify decoding
def verify_decoding(original_sentences, decoded_sentences):
    success = True
    for idx, (orig, decoded) in enumerate(zip(original_sentences, decoded_sentences)):
        if orig == decoded:
            print(f"Sentence {idx+1}: Decoding successful.")
        else:
            print(f"Sentence {idx+1}: Decoding failed.")
            print(f"Original: {orig}")
            print(f"Decoded: {decoded}")
            success = False
    if success:
        print("\nAll sentences decoded correctly!\n")
    else:
        print("\nSome sentences were not decoded correctly.\n")
#calling
verify_decoding(segmented_sentences, decoded_sentences)

Streaming output truncated to the last 5000 lines.
Original: میڈیا اس طرح لوگوں کے ذہنوں میں الجھن پیدا کرتا ھے کہ صرف اور صرف ھماراحکومتی نظام ،ایک ادارے کے علاوہ، معاشرے کے بیگاڑ سبب ھے۔
Decoded: میڈیا اس طرح لوگوں کے ذہنوں میں الجھن پیدا کرتا ھے کہ صرف اور صرف ھماراحکومتی نظام، ایک ادارے کے علاوہ، معاشرے کے بیگاڑ سبب ھے۔
Sentence 78776: Decoding failed.
Original: حال ھی میں آپ نے الیکٹرانک میڈیا کی ایک بڑی شخصیت ، (جس نے اسلام آباد کے سیکٹر آئی 20 کے نا مکمل ھاو ±سنگ پروجیکٹ پر شور و غل ڈالا تھا)
Decoded: حال ھی میں آپ نے الیکٹرانک میڈیا کی ایک بڑی شخصیت،( جس نے اسلام آباد کے سیکٹر آئی 20 کے نا مکمل ھاو ±سنگ پروجیکٹ پر شور و غل ڈالا تھا)
Sentence 78777: Decoding successful.
Sentence 78778: Decoding failed.
Original: اس طرح کا غیر ذمہ دارانہ رویہ گورننس کو تہہ و بالا کر دیتا ھے ، اس طرح لوگ ، جو حکومت کرنے کے آسمانی حق کو یکسر مسترد کرتے ھوۓ اپنے حکومت کرنے کے حق سے رضاکارانہ دستبردار ھوتے ھیں ( اگر آپ حکومتوں کے بننے سے متعلق سوشل کنٹریکٹ کے نظریات کو مانتے ھیں) معاشرتی لا قانونیت ک

In [174]:
def verify_decoding(original_sentences, decoded_sentences):
    successful_decodings = 0
    failed_decodings = 0
    for idx, (orig, decoded) in enumerate(zip(original_sentences, decoded_sentences)):
        if orig == decoded:
            successful_decodings += 1
            print(f"Sentence {idx+1}: Decoding successful.")
        else:
            failed_decodings += 1
            print(f"Sentence {idx+1}: Decoding failed.")
            print(f"Original: {orig}")
            print(f"Decoded: {decoded}")

    print("\nTotal successful decodings:", successful_decodings)
    print("Total failed decodings:", failed_decodings)

#calling
verify_decoding(segmented_sentences, decoded_sentences)


Streaming output truncated to the last 5000 lines.
Original: میڈیا اس طرح لوگوں کے ذہنوں میں الجھن پیدا کرتا ھے کہ صرف اور صرف ھماراحکومتی نظام ،ایک ادارے کے علاوہ، معاشرے کے بیگاڑ سبب ھے۔
Decoded: میڈیا اس طرح لوگوں کے ذہنوں میں الجھن پیدا کرتا ھے کہ صرف اور صرف ھماراحکومتی نظام، ایک ادارے کے علاوہ، معاشرے کے بیگاڑ سبب ھے۔
Sentence 78776: Decoding failed.
Original: حال ھی میں آپ نے الیکٹرانک میڈیا کی ایک بڑی شخصیت ، (جس نے اسلام آباد کے سیکٹر آئی 20 کے نا مکمل ھاو ±سنگ پروجیکٹ پر شور و غل ڈالا تھا)
Decoded: حال ھی میں آپ نے الیکٹرانک میڈیا کی ایک بڑی شخصیت،( جس نے اسلام آباد کے سیکٹر آئی 20 کے نا مکمل ھاو ±سنگ پروجیکٹ پر شور و غل ڈالا تھا)
Sentence 78777: Decoding successful.
Sentence 78778: Decoding failed.
Original: اس طرح کا غیر ذمہ دارانہ رویہ گورننس کو تہہ و بالا کر دیتا ھے ، اس طرح لوگ ، جو حکومت کرنے کے آسمانی حق کو یکسر مسترد کرتے ھوۓ اپنے حکومت کرنے کے حق سے رضاکارانہ دستبردار ھوتے ھیں ( اگر آپ حکومتوں کے بننے سے متعلق سوشل کنٹریکٹ کے نظریات کو مانتے ھیں) معاشرتی لا قانونیت ک

The evaluation function used in this code basically compares the predicted sentence
boundaries with a manually created reference. This helps in measuring the precision, recall,
and F1 score of the segmentation process.

In [161]:
# Evaluation
def evaluate_segmentation(predicted_sentences, reference_sentences):
    def get_boundaries(sentences):
        boundaries = []
        count = 0
        for sentence in sentences:
            count += len(sentence)
            boundaries.append(count)
        return boundaries

    predicted_boundaries= get_boundaries(predicted_sentences)
    reference_boundaries =get_boundaries(reference_sentences)

    predicted_set = set(predicted_boundaries)
    reference_set = set(reference_boundaries)

    TP = len(predicted_set & reference_set)
    FP = len(predicted_set-reference_set)
    FN = len(reference_set- predicted_set)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1_score = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\nEvaluation Metrics:")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")


In [162]:
# manually made small reference segmentation
reference_sentences = [
    "لئے “تبدیلی سرکار” کی چکنی چپڑی باتوں میں آگئی اور اپنے بہتر کہ کھانے کی عادی رہی ہے۔",
    "اسبے چاری عوام چونکہ ہمیشہ سے دھو لئے نئی حکومت کو اقتدار کے ایوانوں تک پہنچا دیا۔ مستقبل کے",
    "پنجاب و بلوچستان میں وزرائے اعلیٰ کی تبدیلی کی لہر میں تیزی آتی جا رہی ہے۔",
    "صوبہ خیبر پختونخوا میں بھی پی ٹی آئی کے سینئرز کے درمیان اختلافات ڈھکے چھپے نہیں۔",
    "پنجاب عملی طور پر غیر اعلانیہ تقسیم کا شکار ہوتا نظر آ رہا ہے۔",
    "اختیارات کی تقسیم نے وسیم اکرم پلس حکومت کو شکایت کرنے پر مجبور کر دیا ہے۔",
    "خاص طور پر پاکستان مسلم لیگ (ق) پنجاب کے انتظامی معاملات پر اپنا اثر و رسوخ بڑھاتی نظر آ رہی ہے۔",
    "وزیرِ اعلیٰ پنجاب تو خود بھی اظہار کر چکے تھے کہ وہ ابھی سیکھ رہے ہیں",
    "اس لیے کہا نہیں جا سکتا کہ اُن کے سیکھنے کا عمل کب مکمل ہوگا۔",
    "حکومتی وزراء کے درمیان لفظی گولہ باری و اختلافات اب کھل کر سامنے آ چکے ہیں۔",
"اگر پنجاب میں تبدیلی کا عمل ہوا تو وفاق بھی اُس کے مضر اثرات سے نہیں بچ سکے گا۔",
"بلوچستان میں حکمراں جماعت کے درمیان جس طرح کھل کر اختلافات سامنے آئے، اس سے خدشات جنم لے رہے ہیں کہ آنے والے دنوں میں ہلچل میں اضافہ ہو سکتا ہے",
"سیاسی انتشار کی وجہ سے صوبے میں دہشت گردی کے خلاف لڑنے والی جنگ بھی متاثر ہو رہی ہے اور انتہاپسندوں کے سہولت کاروں کی سیکیورٹی فورسز کے خلاف کارروائیوں سے ملک و قوم کے محافظوں کو قیمتیجانی نقصان پہنچ رہا ہے۔",
"بلوچستان میں سیاسی ناہمواری کے سبب عوام میں بے چینی کو نمایاں طور پر محسوس کیا جا سکتا ہے۔",

    ]

if reference_sentences:
    evaluate_segmentation(segmented_sentences, reference_sentences)
else:
    print("\nNo reference sentences provided for evaluation.")


Evaluation Metrics:
Precision: 0.0000
Recall: 0.0000
F1 Score: 0.0000


In [175]:
# def segment_paragraph(paragraph):
#     segmented_sentences = urdu_sentence_segmentation(paragraph)
#     return segmented_sentences

# #use function with new input text
# new_paragraph = "گزشتہ کئی سالوں سے مختلف بحران آتے جاتے رہے لیکن حالیہ آٹا ، چینی سمیت دیگر بحران اچانک پید ا ہوئے اور ان پر جے آئی ٹی تشکیل دے دیں گئیں تاکہ عوام کو ریلیف دیا جاسکے دوسری جانب بجلی ، گیس ، پانی سمیت دیگر بلوں میں کئی سو گنا اضافہ کردیا گیا ،صوبائی و وفاقی وزراء نے اپنے اپنے ایوانوں بحران کے ذمہ دار عناصر کو بے نقاب کرنے کے بجائے سب اچھاہے کی رپورٹ پیش کیں ساتھ ہی اورنگزیب کی طرح جواب دیا، حقائق کی ’’قبر ذرا گہری کھودنا!‘‘ تاریخی گواہ ہے بقول ماہرین ، مبصرین اور صحافیوں کا کہنا اور لکھنا ہے کہ پگڑی بدل بھائی کی رسم کی آڑ میں نادر شاہ نے محمد شاہ رنگیلا سے کوہِ نور ہیرا حاصل کیا تھا 12 مئی 1739 کی شام دہلی میں زبردست چہل پہل، شاہجہان آباد میں چراغاں اور لال قلعے میں جشن کا سماں ہے غریبوں میں شربت، پان اور کھانا تقسیم کیا جا رہا ہے، فقیروں، گداؤں کو جھولی بھر بھر کر روپے عطا ہو رہے ہیں آج دربار میں ایرانی بادشاہ نادر شاہ کے سامنے مغلیہ سلطنت کے 13ویں تاجدار محمد شاہ بیٹھے ہیں، لیکن اس وقت ان کے سر پر شاہی تاج نہیں ہے، کیوں نادر شاہ نے ڈھائی ماہ قبل ان سے سلطنت چھین لی تھی 56 دن دہلی میں رہنے کے بعد اب نادر شاہ کے واپس ایران لوٹنے کا وقت آ گیا ہے اور وہ ہندوستان کی باگ ڈور دوبارہ سے محمد شاہ کے حوالے کرنا چاہتا ہے نادر شاہ نے صدیوں سے جمع کردہ مغل خزانے میں جھاڑو پھیر دی ہے اور شہر کے تمام امرا و روسا کی جیبیں الٹا لی ہیں، لیکن اسے دہلی کی ایک طوائف نور بائی نے، جس کا ذکر آگے چل کر آئے گا، خفیہ طور پر بتا دیا ہے کہ یہ سب کچھ جو تم نے حاصل کیا ہے، وہ ایسی چیز کے آگے ہیچ ہے جسے محمد شاہ نے اپنی پگڑی میں چھپا رکھا ہے نادر شاہ گھاگ سیاستدان اور گھاٹ گھاٹ کا پانی پیے ہوئے تھا اس موقعے پر وہ چال چلی جسے نہلے پہ دہلا کہا جاتا ہے اس نے محمد شاہ سے کہا، ’’ایران میں رسم چلی آتی ہے کہ بھائی خوشی کے موقعے پر آپس میں پگڑیاں بدل دیتے ہیں، آج سے ہم بھائی بھائی بن گئے ہیں، تو کیوں نہ اسی رسم کا اعادہ کیا جائے؟‘‘ محمد شاہ کے پاس سر جھکانے کے علاوہ کوئی چارہ نہیں تھا نادر شاہ نے اپنی پگڑی اتار کر اس کے سر رکھی، اور اس کی پگڑی اپنے سر، اور یوں دنیا کا مشہور ترین ہیرا کوہِ نور ہندوستان سے نکل کر ایران پہنچ گیا رنگیلا بادشاہ)اس ہیرے کے مالک محمد شاہ اپنے پڑدادا اورنگزیب عالمگیر کے دورِ حکومت میں 1702 میں پیدا ہوئے تھے ان کا پیدائشی نام تو روشن اختر تھا، تاہم 29 ستمبر 1719 کو بادشاہ گر سید برادران نے انھیں صرف 17 برس کی عمر میں سلطنت تیموریہ کے تخت پر بٹھانے کے بعد ابوالفتح نصیر الدین روشن اختر محمد شاہ کا خطاب دیا خود ان کا تخلص ’’سدا رنگیلا‘‘ تھا اتنا لمبا نام کون یاد رکھتا، چنانچہ عوام نے دونوں کو ملا کر محمد شاہ رنگیلا کر دیا اور وہ آج تک ہندوستان کے طول و عرض میں اسی نام سے جانے اور مانے جاتے ہیں اورنگزیب عالمگیر نے ہندوستان میں ایک خاص قسم کا کٹر اسلام نافذ کر رکھا تھا محمد شاہ کی پیدائش کے وقت اورنگزیب عالمگیر نے ہندوستان میں ایک خاص قسم کا کٹر اسلام نافذ کر رکھا تھا اس کا سب سے پہلا نشانہ وہ فنونِ لطیفہ بنے جن کے بارے میں تصور تھا کہ وہ اسلامی اصولوں سے مطابقت نہیں رکھتے اس کی ایک دلچسپ مثال اطالوی سیاح نکولو منوچی نے لکھی ہے وہ کہتے ہیں کہ اورنگزیبی دور میں جب موسیقی پر پابندی لگی تو گویوں اور موسیقاروں کی روٹی روزی بند ہو گئی آخر تنگ آ کر ایک ہزار فنکاروں نے جمعے کے دن دہلی کی جامع مسجد سے ایک جلوس نکالا اور آلات موسیقی کو جنازوں کی شکل میں لے کر روتے پیٹتے گزرنے لگے اورنگزیب نے دیکھا تو حیرت زدہ ہو کر پچھوایا، یہ کس کا جنازہ لیے جا رہے ہو جس کی خاطر اس قدر آہ و بکا کیا جا رہا ہے؟’’ انھوں نے کہا: ‘‘آپ نے موسیقی قتل کر دی ہے اسے دفنانے جا رہے ہیں اورنگزیب نے جواب دیا، ’’قبر ذرا گہری کھودنا!‘‘طبیعیات کا اصول ہے کہ ہر عمل کا ردِ عمل ہوتا ہے یہی اصول تاریخ اور انسانی معاشرت پر بھی لاگو ہوتا ہے کہ جس چیز کو جتنی سختی سے دبایا جائے، وہ اتنی ہی قوت سے ابھر کر سامنے آتی ہے چنانچہ اورنگزیب کے بعد بھی یہی کچھ ہوا اور محمد شاہ کے دور میں وہ تمام فنون پوری آب و تاب سے سامنے آ گئے جو اس سے پہلے دب گئے تھے 		سب اچھاہے ،حقائق کی قبر ذرا گہری کھودنا"
# segmented_sentences = segment_paragraph(new_paragraph)

# #printing segmented sentences
# print("\nSegmented Sentences from new paragraph:")
# for idx, sentence in enumerate(segmented_sentences):
#     print(f"Sentence {idx+1}: {sentence}")



Segmented Sentences from new paragraph:
Sentence 1: گزشتہ کئی سالوں سے مختلف بحران آتے جاتے رہے
Sentence 2: لیکن حالیہ آٹا ، چینی سمیت دیگر بحران اچانک پید ا ہوئے اور ان پر جے آئی ٹی تشکیل دے دیں گئیں تاکہ عوام کو ریلیف دیا
Sentence 3: جاسکے دوسری جانب بجلی ، گیس ، پانی سمیت دیگر بلوں میں کئی سو گنا اضافہ کردیا گیا ،صوبائی و وفاقی وزراء نے اپنے اپنے ایوانوں بحران کے ذمہ دار عناصر کو بے نقاب کرنے کے بجائے سب اچھاہے کی رپورٹ پیش کیں ساتھ ہی اورنگزیب کی طرح جواب دیا،
Sentence 4: حقائق کی ’’قبر ذرا گہری کھودنا! ‘‘ تاریخی گواہ ہے
Sentence 5: بقول ماہرین ، مبصرین اور صحافیوں کا کہنا اور لکھنا ہے
Sentence 6: کہ پگڑی بدل بھائی کی رسم کی آڑ میں نادر شاہ نے محمد شاہ رنگیلا سے کوہِ نور ہیرا حاصل کیا
Sentence 7: تھا
Sentence 8: 12 مئی 1739 کی شام دہلی میں زبردست چہل پہل، شاہجہان آباد میں چراغاں اور لال قلعے میں جشن کا سماں ہے
Sentence 9: غریبوں میں شربت، پان اور کھانا تقسیم کیا
Sentence 10: جا رہا
Sentence 11: ہے،
Sentence 12: فقیروں، گداؤں کو جھولی بھر بھر کر روپے عطا ہو رہے
Sentence 13: ہیں
Sen

**SentencePiece Tokenizer**

In [176]:
import re
from collections import defaultdict, Counter

#data prepartion
def read_corpus(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        corpus = f.read()
    # Normalizing spaces ans removing unwanted chars
    corpus = re.sub(r'\s+', ' ', corpus)
    return corpus.strip()

Byte Pair Encoding (BPE) algorithm is frequently used in natural language processing to build
a vocabulary from a corpus of texts. This code trains a basic BPE tokenizer with a set
vocabulary size on the corpus.

In [177]:
#tokenizer training using BPE
class BPETokenizer:
    def __init__(self, vocab_size=1000):
        self.vocab_size = vocab_size
        self.vocab = {}
        self.tokens = []

    #vocab
    def get_vocab(self):
        return self.vocab
    # training
    def train(self, corpus):
        tokens = [' '.join(word) + ' </w>' for word in corpus.split(' ')]
        corpus = tokens
        vocab = Counter(corpus)

        while len(self.vocab) < self.vocab_size:
            pairs = self.get_stats(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            self.vocab[best] = pairs[best]
            corpus, vocab = self.merge_vocab(best, corpus, vocab)

        self.tokens = list(self.vocab.keys())

    def get_stats(self, vocab):
        pairs = defaultdict(int)
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols)-1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs

    def merge_vocab(self, pair, corpus, vocab):
        bigram = re.escape(' '.join(pair))
        pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
        # inititliazation
        new_vocab = {}
        new_corpus = []
        for word in corpus:
            w_out = pattern.sub(''.join(pair), word)
            new_corpus.append(w_out)
            new_vocab[w_out] = new_vocab.get(w_out, 0) + vocab[word]
        return new_corpus, new_vocab

    #encoder
    def encode(self, text):
        words = text.strip().split(' ')
        tokens = []
        for word in words:
            word_tokens = self.encode_word(word)
            tokens.extend(word_tokens)
        return tokens

    def encode_word(self, word):
        chars = list(word)
        i = 0
        while i < len(chars):
            j = len(chars)
            while i < j:
                subword = ''.join(chars[i:j])
                if subword in self.tokens:
                    yield subword
                    i =j
                    break
                else:
                    j -=1
            else:
                yield chars[i]
                i+=1

    #decoder
    def decode(self, tokens):
        text = ''.join(tokens)
        return text

In [178]:
if __name__ == '__main__':
    corpus_text = text

    #training tokenizer
    bpe_tokenizer = BPETokenizer(vocab_size=50)

    bpe_tokenizer.train(corpus_text)

    #vocabulary
    vocab = bpe_tokenizer.get_vocab()
    print("vocabulary:")
    for token, freq in vocab.items():
        print(f"{token}: {freq}")

    #sample text
    sample_text = "فقیروں، گداؤں کو جھولی بھر بھر کر روپے عطا ہو رہے"
    encoded_tokens = list(bpe_tokenizer.encode(sample_text))
    print("Encoded Tokens:")
    print(encoded_tokens)

    #decoding back to text
    decoded_text = bpe_tokenizer.decode(encoded_tokens)
    print("Decoded Text:")
    print(decoded_text)


vocabulary:
('ے', '</w>'): 243346
('ک', 'ے</w>'): 2508558540
('ی', '</w>'): 63648794274659
('ک', 'ی</w>'): 2395449686095959408
('ں', '</w>'): 46937656194769860826552
('ی', 'ں</w>'): 1590151010765956921964515750
('م', 'یں</w>'): 54033925346342074612172839353644
('ہ', 'ے</w>'): 238398266986696458759639084053094196
('س', 'ے</w>'): 3119106133589270454291515835289397342123
('ر', '</w>'): 55889573142067745799509741326727370508462257
('و', 'ر</w>'): 1320136879286357745367881695607619125868925621459
('ا', 'ور</w>'): 31276682944051934050576083619111890525554902523396768
('و', '</w>'): 122677265490257359866118114675800260283265343234654050383
('ک', 'و</w>'): 2530951988968545181062724328898064500323340093457498363571255
('ا', '</w>'): 13565461187006689946662663300390876440500437143112602669276773254
('ک', 'ا</w>'): 255817412811203945980587736119038277516512803257387997366543311409488
('ہ', '</w>'): 1700400299954397970192274803008618588820243722675073607784951131211033211
('ک', 'ہ</w>'): 301579795